# AO3 EDA Notebook: `works` and `kudos`

This notebook performs exploratory data analysis on the `works` and `kudos` tables in `data/ao3.db`.

Each section briefly states the calculation or visualization being produced; it does not attempt to interpret outcomes.


In [1]:
print("Hello")

Hello


In [2]:
# Install dependencies if needed (safe to re-run).
import importlib.util
import subprocess
import sys

required = ["pandas", "numpy", "matplotlib", "seaborn"]
missing = [pkg for pkg in required if importlib.util.find_spec(pkg) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

print("Missing packages installed:", missing if missing else "None")


Missing packages installed: None


In [ ]:
import ast
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 12

DB_PATH = Path("data/ao3.db")
if not DB_PATH.exists():
    raise FileNotFoundError(f"Database not found at {DB_PATH.resolve()}")

conn = sqlite3.connect(DB_PATH)
print("Connected to:", DB_PATH.resolve())


FileNotFoundError: Database not found at C:\Users\buech\OneDrive\Documents\MATLAB\AO3 Scraping\notebooks\data\ao3.db

## 1) Database Inventory

This section lists tables, row counts, and column definitions to confirm data availability and structure before analysis.


In [ ]:
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn)
tables


In [ ]:
row_counts = pd.concat(
    [pd.read_sql_query(f"SELECT '{t}' AS table_name, COUNT(*) AS rows FROM {t}", conn) for t in ["works", "kudos"]],
    ignore_index=True,
)
row_counts


In [ ]:
works_schema = pd.read_sql_query("PRAGMA table_info(works)", conn)
kudos_schema = pd.read_sql_query("PRAGMA table_info(kudos)", conn)

display(works_schema)
display(kudos_schema)


## 2) Load and Prepare `works`

This section loads all rows from `works`, coerces date/time fields, and creates helper columns used in plots and summaries.


In [ ]:
works = pd.read_sql_query("SELECT * FROM works", conn)

# Parse AO3-style date strings like '31 Oct 2025'.
works["updated_date_parsed"] = pd.to_datetime(works["updated_date"], format="%d %b %Y", errors="coerce")
works["scraped_at_utc_parsed"] = pd.to_datetime(works["scraped_at_utc"], utc=True, errors="coerce")

works["updated_year"] = works["updated_date_parsed"].dt.year
works["updated_month"] = works["updated_date_parsed"].dt.to_period("M").astype(str)

works.shape


In [ ]:
works.head(3)


## 3) `works` Data Completeness and Types

The calculations below show missing-value rates and data types for core fields in `works`.


In [ ]:
dtype_summary = pd.DataFrame({
    "column": works.columns,
    "dtype": works.dtypes.astype(str).values,
    "non_null": works.notna().sum().values,
    "nulls": works.isna().sum().values,
    "null_pct": (works.isna().mean().values * 100).round(2),
}).sort_values("null_pct", ascending=False)
dtype_summary


In [ ]:
null_pct = works.isna().mean().sort_values(ascending=False)
null_pct = null_pct[null_pct > 0]

if len(null_pct) == 0:
    print("No missing values detected in works.")
else:
    ax = null_pct.plot(kind="bar", color="#4C72B0")
    ax.set_title("Missingness Rate by Column (works)")
    ax.set_ylabel("Fraction missing")
    plt.xticks(rotation=65, ha="right")
    plt.tight_layout()
    plt.show()


## 4) Numeric Field Distributions in `works`

These histograms and boxplots summarize spread, central tendency, and tail behavior for numeric metrics.


In [ ]:
numeric_cols = [
    c for c in [
        "summary_len", "words", "chapters_current", "chapters_total",
        "comments", "kudos", "bookmarks", "hits", "guest_kudos"
    ] if c in works.columns
]

works[numeric_cols].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T


In [ ]:
fig, axes = plt.subplots(nrows=int(np.ceil(len(numeric_cols) / 3)), ncols=3, figsize=(15, 4 * int(np.ceil(len(numeric_cols) / 3))))
axes = np.array(axes).reshape(-1)

for i, col in enumerate(numeric_cols):
    vals = works[col].dropna()
    sns.histplot(vals, bins=40, ax=axes[i], color="#55A868")
    axes[i].set_title(f"Distribution: {col}")

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
heavy_tail_cols = [c for c in ["words", "comments", "kudos", "bookmarks", "hits", "guest_kudos"] if c in works.columns]

fig, axes = plt.subplots(nrows=int(np.ceil(len(heavy_tail_cols) / 3)), ncols=3, figsize=(15, 4 * int(np.ceil(len(heavy_tail_cols) / 3))))
axes = np.array(axes).reshape(-1)

for i, col in enumerate(heavy_tail_cols):
    vals = np.log1p(works[col].fillna(0))
    sns.histplot(vals, bins=40, ax=axes[i], color="#C44E52")
    axes[i].set_title(f"log1p Distribution: {col}")

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()


## 5) Categorical Composition in `works`

These bar charts show frequency distributions for key categorical metadata fields.


In [ ]:
cat_cols = [c for c in ["rating", "warning_summary", "category", "completion_status", "language"] if c in works.columns]

for col in cat_cols:
    top_n = 20 if col in {"warning_summary", "category", "language"} else 10
    vc = works[col].fillna("<MISSING>").value_counts().head(top_n)
    plt.figure(figsize=(10, 5))
    sns.barplot(x=vc.values, y=vc.index, orient="h", color="#8172B2")
    plt.title(f"Top {top_n} Values: {col}")
    plt.xlabel("Count")
    plt.ylabel(col)
    plt.tight_layout()
    plt.show()


## 6) Date-Based Views for `works`

These plots aggregate records by updated date and scrape timestamp to show volume over time.


In [ ]:
works_by_year = works.dropna(subset=["updated_year"]).groupby("updated_year", as_index=False).size()
works_by_year.columns = ["updated_year", "works_count"]

plt.figure(figsize=(10, 5))
sns.lineplot(data=works_by_year, x="updated_year", y="works_count", marker="o", color="#CCB974")
plt.title("Works Count by Updated Year")
plt.xlabel("Updated year")
plt.ylabel("Number of works")
plt.tight_layout()
plt.show()


In [ ]:
monthly_counts = (
    works.dropna(subset=["updated_date_parsed"])
    .set_index("updated_date_parsed")
    .resample("MS")
    .size()
    .rename("works_count")
    .reset_index()
)

plt.figure(figsize=(12, 5))
sns.lineplot(data=monthly_counts, x="updated_date_parsed", y="works_count", color="#64B5CD")
plt.title("Works Count by Updated Month")
plt.xlabel("Updated month")
plt.ylabel("Number of works")
plt.tight_layout()
plt.show()


In [ ]:
scrape_hour = works["scraped_at_utc_parsed"].dt.hour
hourly = scrape_hour.value_counts().sort_index()

plt.figure(figsize=(10, 4))
sns.barplot(x=hourly.index, y=hourly.values, color="#8C8C8C")
plt.title("works Rows by Scrape Hour (UTC)")
plt.xlabel("Hour of day (UTC)")
plt.ylabel("Rows")
plt.tight_layout()
plt.show()


## 7) Fandom List Analysis from `fandoms_json`

This section parses the serialized fandom list, computes list lengths per work, and shows top fandom frequencies after exploding lists.


In [ ]:
def parse_fandoms(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    text = str(value).strip()
    if not text:
        return []
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, list):
            return [str(x).strip() for x in parsed if str(x).strip()]
    except (ValueError, SyntaxError):
        pass
    return [text]

works["fandoms_list"] = works["fandoms_json"].apply(parse_fandoms)
works["fandom_count"] = works["fandoms_list"].apply(len)

works["fandom_count"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99])


In [ ]:
plt.figure(figsize=(10, 4))
sns.histplot(works["fandom_count"], bins=30, color="#937860")
plt.title("Distribution of Number of Fandoms per Work")
plt.xlabel("Fandom count")
plt.ylabel("Works")
plt.tight_layout()
plt.show()


In [ ]:
top_fandoms = (
    works[["work_id", "fandoms_list"]]
    .explode("fandoms_list")
    .dropna(subset=["fandoms_list"])
    .assign(fandom=lambda d: d["fandoms_list"].astype(str).str.strip())
    .query("fandom != ''")
    .groupby("fandom", as_index=False)
    .size()
    .sort_values("size", ascending=False)
    .head(30)
)

display(top_fandoms.head(15))

plt.figure(figsize=(11, 8))
sns.barplot(data=top_fandoms, x="size", y="fandom", color="#DA8BC3")
plt.title("Top 30 Fandom Labels Across Works")
plt.xlabel("Occurrences across works")
plt.ylabel("Fandom")
plt.tight_layout()
plt.show()


## 8) Derived Engagement Ratios in `works`

These calculations build normalized metrics (for example, kudos per hit) and visualize their distributions.


In [ ]:
ratio_df = works.copy()

ratio_df["kudos_per_hit"] = ratio_df["kudos"] / ratio_df["hits"].replace(0, np.nan)
ratio_df["bookmarks_per_hit"] = ratio_df["bookmarks"] / ratio_df["hits"].replace(0, np.nan)
ratio_df["comments_per_kudos"] = ratio_df["comments"] / ratio_df["kudos"].replace(0, np.nan)
ratio_df["guest_kudos_share"] = ratio_df["guest_kudos"] / ratio_df["kudos"].replace(0, np.nan)

ratio_cols = ["kudos_per_hit", "bookmarks_per_hit", "comments_per_kudos", "guest_kudos_share"]
ratio_df[ratio_cols].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.reshape(-1)

for i, col in enumerate(["kudos_per_hit", "bookmarks_per_hit", "comments_per_kudos", "guest_kudos_share"]):
    vals = ratio_df[col].replace([np.inf, -np.inf], np.nan).dropna()
    vals = vals.clip(upper=vals.quantile(0.99) if len(vals) else None)
    sns.histplot(vals, bins=40, ax=axes[i], color="#4C72B0")
    axes[i].set_title(f"Distribution: {col} (capped at p99)")

plt.tight_layout()
plt.show()


## 9) Correlations Among Numeric `works` Metrics

The heatmap below computes pairwise Spearman correlations across numeric fields and derived ratio features.


In [ ]:
corr_cols = [c for c in [
    "summary_len", "words", "chapters_current", "chapters_total",
    "comments", "kudos", "bookmarks", "hits", "guest_kudos",
    "kudos_per_hit", "bookmarks_per_hit", "comments_per_kudos", "guest_kudos_share"
] if c in ratio_df.columns]

corr = ratio_df[corr_cols].corr(method="spearman")

plt.figure(figsize=(12, 9))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False, square=True)
plt.title("Spearman Correlation Heatmap (works metrics)")
plt.tight_layout()
plt.show()

corr


## 10) `kudos` Table Coverage and Scale

This section computes table-level counts and join coverage between `kudos` and `works`.


In [ ]:
kudos_overview = pd.read_sql_query(
    """
    SELECT
        (SELECT COUNT(*) FROM kudos) AS kudos_rows,
        (SELECT COUNT(DISTINCT work_id) FROM kudos) AS kudos_distinct_work_ids,
        (SELECT COUNT(*) FROM works) AS works_rows,
        (SELECT COUNT(*) FROM works w WHERE w.work_id IN (SELECT DISTINCT work_id FROM kudos)) AS works_with_any_kudos_row
    """,
    conn,
)

kudos_overview


## 11) Per-Work Aggregation from `kudos`

These calculations aggregate the `kudos` rows at work level and plot the resulting distributions.


In [ ]:
kudos_per_work = pd.read_sql_query(
    """
    SELECT
        work_id,
        COUNT(*) AS kudos_rows,
        COUNT(DISTINCT username) AS distinct_users,
        COUNT(DISTINCT pseud_url) AS distinct_pseuds
    FROM kudos
    GROUP BY work_id
    """,
    conn,
)

kudos_per_work.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for i, col in enumerate(["kudos_rows", "distinct_users", "distinct_pseuds"]):
    vals = np.log1p(kudos_per_work[col])
    sns.histplot(vals, bins=40, ax=axes[i], color="#55A868")
    axes[i].set_title(f"log1p Distribution: {col}")

plt.tight_layout()
plt.show()


## 12) User-Level Activity in `kudos`

This section computes how many distinct works each user has left kudos on, then visualizes top users and the user activity distribution.


In [ ]:
top_users = pd.read_sql_query(
    """
    SELECT
        username,
        COUNT(DISTINCT work_id) AS works_kudoed,
        COUNT(*) AS kudos_rows
    FROM kudos
    GROUP BY username
    ORDER BY works_kudoed DESC, kudos_rows DESC
    LIMIT 30
    """,
    conn,
)

user_activity_dist = pd.read_sql_query(
    """
    SELECT
        works_kudoed,
        COUNT(*) AS user_count
    FROM (
        SELECT username, COUNT(DISTINCT work_id) AS works_kudoed
        FROM kudos
        GROUP BY username
    )
    GROUP BY works_kudoed
    ORDER BY works_kudoed
    """,
    conn,
)

display(top_users.head(15))
display(user_activity_dist.head(15))


In [ ]:
plt.figure(figsize=(11, 8))
sns.barplot(data=top_users.sort_values("works_kudoed", ascending=True), x="works_kudoed", y="username", color="#C44E52")
plt.title("Top 30 Users by Distinct Works Kudoed")
plt.xlabel("Distinct works kudoed")
plt.ylabel("Username")
plt.tight_layout()
plt.show()


In [ ]:
dist_plot = user_activity_dist.copy()
dist_plot = dist_plot[dist_plot["works_kudoed"] <= dist_plot["works_kudoed"].quantile(0.99)]

plt.figure(figsize=(10, 5))
sns.lineplot(data=dist_plot, x="works_kudoed", y="user_count", marker="o", color="#8172B2")
plt.title("User Count by Number of Distinct Works Kudoed (trimmed at p99)")
plt.xlabel("Distinct works kudoed by user")
plt.ylabel("Number of users")
plt.tight_layout()
plt.show()


## 13) `kudos` Scrape Timeline

These plots summarize ingestion timing by day and hour from `kudos.scraped_at_utc`.


In [ ]:
kudos_time = pd.read_sql_query(
    """
    SELECT scraped_at_utc
    FROM kudos
    """,
    conn,
)

kudos_time["scraped_at_utc"] = pd.to_datetime(kudos_time["scraped_at_utc"], utc=True, errors="coerce")
kudos_time["scrape_day"] = kudos_time["scraped_at_utc"].dt.floor("D")
kudos_time["scrape_hour"] = kudos_time["scraped_at_utc"].dt.hour

kudos_daily = kudos_time.groupby("scrape_day", as_index=False).size().rename(columns={"size": "rows"})
kudos_hourly = kudos_time.groupby("scrape_hour", as_index=False).size().rename(columns={"size": "rows"})

display(kudos_daily)
display(kudos_hourly)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.barplot(data=kudos_daily, x="scrape_day", y="rows", ax=axes[0], color="#CCB974")
axes[0].set_title("kudos Rows by Scrape Day (UTC)")
axes[0].set_xlabel("Day")
axes[0].set_ylabel("Rows")
axes[0].tick_params(axis="x", rotation=45)

sns.barplot(data=kudos_hourly, x="scrape_hour", y="rows", ax=axes[1], color="#64B5CD")
axes[1].set_title("kudos Rows by Scrape Hour (UTC)")
axes[1].set_xlabel("Hour")
axes[1].set_ylabel("Rows")

plt.tight_layout()
plt.show()


## 14) Join `works` with Aggregated `kudos`

This section joins per-work `kudos` aggregates with `works` metrics and compares stored counters with observed row counts.


In [ ]:
works_kudos_join = works[["work_id", "title", "kudos", "guest_kudos", "hits", "comments", "bookmarks", "words"]].merge(
    kudos_per_work, on="work_id", how="left"
)

works_kudos_join[["kudos_rows", "distinct_users", "distinct_pseuds"]] = works_kudos_join[["kudos_rows", "distinct_users", "distinct_pseuds"]].fillna(0)

works_kudos_join["kudos_minus_rows"] = works_kudos_join["kudos"] - works_kudos_join["kudos_rows"]
works_kudos_join["guest_share_from_rows"] = works_kudos_join["guest_kudos"] / works_kudos_join["kudos_rows"].replace(0, np.nan)

works_kudos_join.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T


In [ ]:
plt.figure(figsize=(7, 6))
sample = works_kudos_join.sample(min(5000, len(works_kudos_join)), random_state=42)
sns.scatterplot(data=sample, x="kudos", y="kudos_rows", alpha=0.35, s=18, color="#4C72B0")
max_axis = np.nanmax([sample["kudos"].max(), sample["kudos_rows"].max()])
plt.plot([0, max_axis], [0, max_axis], color="black", linestyle="--", linewidth=1)
plt.title("Stored `works.kudos` vs. Aggregated `kudos` Row Count")
plt.xlabel("works.kudos")
plt.ylabel("kudos table row count by work")
plt.tight_layout()
plt.show()


In [ ]:
delta = works_kudos_join["kudos_minus_rows"].dropna()

plt.figure(figsize=(10, 4))
sns.histplot(delta.clip(lower=delta.quantile(0.01), upper=delta.quantile(0.99)), bins=50, color="#C44E52")
plt.title("Distribution of `works.kudos - kudos_rows` (trimmed to p1-p99)")
plt.xlabel("Difference")
plt.ylabel("Works")
plt.tight_layout()
plt.show()

delta.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])


## 15) Basic Integrity Checks

These queries flag duplicates, missing join coverage, and selected metric consistency conditions.


In [ ]:
checks = {}

checks["works_duplicate_work_id_rows"] = pd.read_sql_query(
    "SELECT COUNT(*) AS n FROM (SELECT work_id, COUNT(*) c FROM works GROUP BY work_id HAVING c > 1)", conn
).iloc[0, 0]

checks["kudos_rows_without_matching_work"] = pd.read_sql_query(
    "SELECT COUNT(*) AS n FROM kudos k LEFT JOIN works w ON k.work_id = w.work_id WHERE w.work_id IS NULL", conn
).iloc[0, 0]

checks["works_without_any_kudos_row"] = pd.read_sql_query(
    "SELECT COUNT(*) AS n FROM works w LEFT JOIN (SELECT DISTINCT work_id FROM kudos) k ON w.work_id = k.work_id WHERE k.work_id IS NULL", conn
).iloc[0, 0]

checks["works_where_kudos_gt_hits"] = works_kudos_join.query("kudos > hits").shape[0]
checks["works_where_guest_kudos_gt_kudos"] = works_kudos_join.query("guest_kudos > kudos").shape[0]

pd.Series(checks, name="count").to_frame()


In [ ]:
print("EDA notebook complete.")
print("Close the SQLite connection if you are done exploring.")


In [ ]:
# Optional cleanup
# conn.close()
